# MicrobiomeDataSpace and MAGraph



In [1]:
import sys; print(sys.executable)

/opt/conda/bin/python


In [ ]:
! sh ./mdo_workshop_setup.sh # > /dev/null 2>&1 # take < 3 min for run
# then select the new kernel named metabiome

conda-forge/linux-aarch64                                   Using cache
conda-forge/noarch                                          Using cache
[+] 0.0s

Pinned packages:

  - python=3.10


Transaction

  Prefix: /opt/conda

  All requested packages already installed


Transaction starting
[+] 0.0s

Transaction finished

Hit:1 http://ports.ubuntu.com/ubuntu-ports noble InRelease
Hit:2 http://ports.ubuntu.com/ubuntu-ports noble-updates InRelease
Hit:3 http://ports.ubuntu.com/ubuntu-ports noble-backports InRelease
Hit:4 http://ports.ubuntu.com/ubuntu-ports noble-security InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.12-dev is already the newest version (3.12.3-1ubuntu0.8).
build-essential is already the newest version (12.10ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 4 not upgraded.
error: unexpected argument 'uv' found

Usage: uv sync [OPTIONS]

For more information, try '--he

## An overview

*   

## Setup of the environment



In [1]:
import polars as pl
import metabiome.io as mio

## Load data

In [2]:
data_dir = "/biodata/resources/day3_lab5/nar_operon_query_output"
# Load from multiple file formats
mds = mio.from_files(
    obs="{}/input/metadata".format(data_dir),           # Sample metadata
    abundance="{}/input/RPKM.json".format(data_dir),   # Gene abundance profiles
    taxonomy="{}/input/taxonomy.json".format(data_dir), # Taxonomic annotations
    functional="{}/input/FG.json".format(data_dir),    # Functional groups
    sequences="{}/input/merged.fasta".format(data_dir), # Gene sequences
    id_mapping="{}/input/id_mapping.tsv".format(data_dir) # ID cross-references
)


In [3]:
# Check data dimensions
print(f"Data shape: {mds.shape}")  # (n_samples, n_features)

Data shape: (9522, 9484)


## Profile the abundance per species per sample


In [4]:
# aggregrate by pfam_domain and taxa first
operon_abd_data = mds.groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_diseaseMean = operon_abd_data.groupby.obs("disease_group").agg("mean")


In [13]:
operon_abd_data.var

species,pfam_domain,gc_id,msp_id,gene_category,domain,kingdom,phylum,class,order,family,genus,var
str,str,str,str,str,str,str,str,str,str,str,str,str
"""uncultured Actinomyces sp.""","""PF02665""","""Israel__10128__k141_19803::3::…","""msp_0582""","""shared_core""","""Bacteria""","""Bacillati""","""Actinomycetota""","""Actinomycetes""","""Actinomycetales""","""Actinomycetaceae""","""Actinomyces""","""uncultured Actinomyces sp."""
"""Kluyvera georgiana""","""PF13247:::PF14711""","""cohort_merged__ERR2619749__k99…","""msp_010""","""core""","""Bacteria""","""Pseudomonadati""","""Pseudomonadota""","""Gammaproteobacteria""","""Enterobacterales""","""Enterobacteriaceae""","""Kluyvera""","""Kluyvera georgiana"""
"""Veillonella dispar""","""PF14710:::PF00384:::PF01568""","""cohort_merged__SRR5650035__k99…","""msp_280""","""shared_core""","""Bacteria""","""Bacillati""","""Bacillota""","""Negativicutes""","""Veillonellales""","""Veillonellaceae""","""Veillonella""","""Veillonella dispar"""
"""Kluyvera cryocrescens""","""PF02665""","""France__9828__k141_17695::1::5…","""msp_0796""","""core""","""Bacteria""","""Pseudomonadati""","""Pseudomonadota""","""Gammaproteobacteria""","""Enterobacterales""","""Enterobacteriaceae""","""Kluyvera""","""Kluyvera cryocrescens"""
"""Leclercia adecarboxylata""","""PF14710:::PF00384:::PF01568""","""cohort_merged__ERR2619749__k99…","""msp_011""","""core""","""Bacteria""","""Pseudomonadati""","""Pseudomonadota""","""Gammaproteobacteria""","""Enterobacterales""","""Enterobacteriaceae""","""Leclercia""","""Leclercia adecarboxylata"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""Parabacteroides merdae""","""PF02613""","""cohort_merged__SRR5650178__k99…","""msp_012""","""shared_accessory""","""Bacteria""","""Pseudomonadati""","""Bacteroidota""","""Bacteroidia""","""Bacteroidales""","""Tannerellaceae""","""Parabacteroides""","""Parabacteroides merdae"""
"""uncultured Slackia sp.""","""PF02613""","""US__11552__k141_361145::29::35…","""msp_1435""","""shared_core""","""Bacteria""","""Bacillati""","""Actinomycetota""","""Coriobacteriia""","""Eggerthellales""","""Eggerthellaceae""","""Slackia""","""uncultured Slackia sp."""
"""Rothia dentocariosa""","""PF02665""","""France__9820__k141_42146::3::2…","""msp_1581""","""core""","""Bacteria""","""Bacillati""","""Actinomycetota""","""Actinomycetes""","""Micrococcales""","""Micrococcaceae""","""Rothia""","""Rothia dentocariosa"""


In [ ]:
cur_species ="Veillonella parvula" #"Veillonella parvula", "Escherichia coli"
operon_abd_data.pl.boxplot(
    feature_name=cur_species, 
    x_axis_col="disease_group",
    feature_type_col="species",
    title="Abundance of {0} by Disease Group".format(cur_species),
    color_map = {
            "Healthy": "#2166ac",
            # "nonIBD": "#2166ac",
            "CD": "#b2182b",
            "UC": "#d6604d",
            # "CRC": "#762a83",
            # "MP": "#9970ab",
            # "adenoma": "#c2a5cf",
        }
)



## Stratify by disease group

In [7]:
# operon_abd_data_CD = operon_abd_data.filter.obs(pl.col("disease_group") == "CD")
operon_abd_data_CD = mds.filter.obs(pl.col("disease_group") == "CD").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_UC = mds.filter.obs(pl.col("disease_group") == "UC").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_Healthy = mds.filter.obs(pl.col("disease_group") == "Healthy").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")


/workspaces/course-pilot/Metabiome/src/metabiome/core/space.py:369: UserWarning:

1 filtered samples were not present in the matrix and were dropped



In [9]:
operon_abd_data_Healthy.pl.sankey(
    hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.5
)


In [10]:
operon_abd_data_CD = mds.filter.obs(pl.col("disease_group") == "CD").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_CD.pl.sankey(
   hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.5
)

In [11]:
operon_abd_data_UC = mds.filter.obs(pl.col("disease_group") == "UC").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_UC.pl.sankey(
   hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.5
)

/workspaces/course-pilot/Metabiome/src/metabiome/core/space.py:369: UserWarning:

1 filtered samples were not present in the matrix and were dropped



### Task: 

